# 02 — Collaboration transmits optimism: beliefs converge

Reproduces the baseline-continuity (disruption) model and the convergence null tests (permutation + pseudo-group), for related and unrelated items. Permutations use 10,000 resamples.

**Expected:** 3-way pre×condition×topic β=−.135, p=.019; related slopes .635 (individual) vs .449 (collaborative), difference β=−.186, SE=.045, 95%CI[−.274, −.098], p<.001; unrelated slopes .699 vs .648, difference β=−.050, SE=.040, 95%CI[−.128, +.028], p=.20. Convergence (related): observed −2.39; permutation null −0.08 (SD 1.11), p=.022; pseudo-group −0.14 (SD 0.87), p=.007. Unrelated: −1.99; permutation −1.40, p=.22; pseudo −2.30, p=.68.

In [1]:
import pandas as pd, numpy as np, warnings
from scipy import stats
import statsmodels.formula.api as smf
warnings.filterwarnings('ignore')
DATA = '../data'
def load_belief(): return pd.read_excel(f'{DATA}/belief_items.xlsx')
def load_subject(): return pd.read_excel(f'{DATA}/subject_measures.xlsx')
def load_group(): return pd.read_csv(f'{DATA}/group_conversation.csv')
def rep(m,t):
    ci=m.conf_int().loc[t]
    return f'b={m.params[t]:+.3f} 95%CI[{ci[0]:+.3f}, {ci[1]:+.3f}] p={m.pvalues[t]:.4f}'
SEED=20260725; NP=10000


In [2]:
b=load_belief().dropna(subset=['pre','post']).copy()
# disruption model: post ~ pre * condition * topic, RI participant
b['topic_n']=(b.topic=='related').astype(int); b['cond']=pd.Categorical(b.condition,['individual','collaborative'])
m=smf.mixedlm('post ~ pre * cond * topic_n', b, groups=b.participant).fit()
P=m.fe_params; cov=m.cov_params().loc[list(P.index),list(P.index)].values; fe=list(P.index)
def slope(active):
    v=np.zeros(len(fe))
    for t,c in active.items(): v[fe.index(t)]=c
    return float(v@P.values), float(np.sqrt(v@cov@v))
print('3-way pre:cond:topic:', rep(m,'pre:cond[T.collaborative]:topic_n'))
t3='pre:cond[T.collaborative]:topic_n'
print(f'   SE={m.bse[t3]:.3f}, z={m.tvalues[t3]:.3f}')
for cond,cc in [('individual',{}),('collaborative',{'pre:cond[T.collaborative]':1})]:
    est,se=slope({'pre':1,**cc,**({'pre:topic_n':1,**({'pre:cond[T.collaborative]:topic_n':1} if cc else {})})})
    print(f'  related slope {cond:13s} = {est:.3f}, SE={se:.3f}, 95%CI[{est-1.96*se:.3f}, {est+1.96*se:.3f}]')
for cond,cc in [('individual',{'pre':1}),('collaborative',{'pre':1,'pre:cond[T.collaborative]':1})]:
    est,se=slope(cc); print(f'  unrelated slope {cond:13s} = {est:.3f}, SE={se:.3f}, 95%CI[{est-1.96*se:.3f}, {est+1.96*se:.3f}]')
def diff(active, lab):
    est,se=slope(active); p=2*stats.norm.sf(abs(est/se))
    print(f'  {lab} collab-indiv diff = {est:+.3f}, SE={se:.3f}, 95%CI[{est-1.96*se:+.3f}, {est+1.96*se:+.3f}], p={p:.3g}')
diff({'pre:cond[T.collaborative]':1,'pre:cond[T.collaborative]:topic_n':1}, 'related  ')
diff({'pre:cond[T.collaborative]':1}, 'unrelated')


3-way pre:cond:topic: b=-0.135 95%CI[-0.249, -0.022] p=0.0198
   SE=0.058, z=-2.331
  related slope individual    = 0.635, SE=0.032, 95%CI[0.573, 0.697]
  related slope collaborative = 0.449, SE=0.032, 95%CI[0.387, 0.511]
  unrelated slope individual    = 0.699, SE=0.028, 95%CI[0.643, 0.754]
  unrelated slope collaborative = 0.648, SE=0.028, 95%CI[0.593, 0.703]
  related   collab-indiv diff = -0.186, SE=0.045, 95%CI[-0.274, -0.098], p=3.32e-05
  unrelated collab-indiv diff = -0.050, SE=0.040, 95%CI[-0.128, +0.028], p=0.205


### Within-group distance, pre vs post (related items)

For each collaborative participant: the mean absolute difference between their related-item ratings and those of each other member of their group, averaged over group-mates; pre vs post compared with a paired t-test (Cohen's d_z).

In [3]:
e=b[(b.condition=='collaborative')&(b.topic=='related')]
rows=[]
for g,d in e.groupby('group'):
    pre=d.pivot_table('pre','Item','participant'); post=d.pivot_table('post','Item','participant')
    for p in pre.columns:
        o=[q for q in pre.columns if q!=p]
        rows.append(dict(participant=p,
                         pre=np.mean([np.nanmean(np.abs(pre[p]-pre[q])) for q in o]),
                         post=np.mean([np.nanmean(np.abs(post[p]-post[q])) for q in o])))
dd=pd.DataFrame(rows); diff_=dd.pre-dd.post
t,p=stats.ttest_rel(dd.pre,dd.post)
print(f'n={len(dd)} | distance pre M={dd.pre.mean():.2f}, post M={dd.post.mean():.2f} | paired t({len(dd)-1})={t:.3f}, p={p:.4f}, d_z={diff_.mean()/diff_.std(ddof=1):.3f}')

n=76 | distance pre M=26.27, post M=23.88 | paired t(75)=2.437, p=0.0172, d_z=0.280


In [4]:
# convergence null tests (optimized: precompute pairwise distance matrices)
def pdist(mat):
    n=mat.shape[1]; D=np.zeros((n,n))
    for i in range(n): D[i]=np.nanmean(np.abs(mat.T-mat[:,i]),axis=1)
    return D
def conv(gidx,Dpre,Dpost):
    ds=[]
    for m2 in gidx:
        if len(m2)<2: continue
        for p in m2:
            o=[q for q in m2 if q!=p]; ds.append(np.mean(Dpost[p,o])-np.mean(Dpre[p,o]))
    return np.mean(ds)
def test(topic,rng):
    e=b[(b.condition=='collaborative')&(b.topic==topic)]; c=b[(b.condition=='individual')&(b.topic==topic)]
    pe=e.pivot_table('pre','Item','participant'); po=e.pivot_table('post','Item','participant')
    pidx={p:i for i,p in enumerate(pe.columns)}; Dpre=pdist(pe.values); Dpost=pdist(po.values)
    real=[[pidx[p] for p in v] for v in e.groupby('group')['participant'].unique()]; sizes=[len(x) for x in real]
    obs=conv(real,Dpre,Dpost)
    idx=np.arange(len(pe.columns)); perm=np.empty(NP)
    for k in range(NP):
        pp=rng.permutation(idx); i=0; gg=[]
        for s in sizes: gg.append(list(pp[i:i+s])); i+=s
        perm[k]=conv(gg,Dpre,Dpost)
    cpe=c.pivot_table('pre','Item','participant'); cpo=c.pivot_table('post','Item','participant')
    cDpre=pdist(cpe.values); cDpost=pdist(cpo.values); cn=cpe.shape[1]; cidx=np.arange(cn); ps=np.empty(NP)
    for k in range(NP):
        pp=rng.permutation(cidx); i=0; gg=[]
        for s in sizes:
            if i+s<=cn: gg.append(list(pp[i:i+s])); i+=s
        ps[k]=conv(gg,cDpre,cDpost)
    print(f'{topic:9s} observed={obs:+.2f} | permutation null {perm.mean():+.2f} (SD {perm.std():.2f}) p={(np.sum(perm<=obs)+1)/(NP+1):.3f}'
          f' | pseudo-group {ps.mean():+.2f} (SD {ps.std():.2f}) p={(np.sum(ps<=obs)+1)/(NP+1):.3f}')
rng=np.random.default_rng(SEED)
for tp in ['related','unrelated']: test(tp,rng)


related   observed=-2.39 | permutation null -0.08 (SD 1.11) p=0.022 | pseudo-group -0.14 (SD 0.87) p=0.007
unrelated observed=-1.99 | permutation null -1.38 (SD 0.81) p=0.223 | pseudo-group -2.30 (SD 0.64) p=0.683
